# Getting Started

## What is it?
Spark is a distributed compute engine that lets one program describe work over data too large or slow for a single-process workflow.

## Why does it exist?
Spark lets a single program express parallel, fault-tolerant data work. Its optimizer can inspect DataFrame/SQL expressions before running them.

## When should I use it?
Use this technique when the data relationship or operational concern in the examples matches your problem; validate assumptions with data profiles and plans.

## How does it work?
```mermaid
flowchart LR
  A[DataFrame transformations] --> B[Logical plan]
  B --> C[Catalyst / physical plan]
  C --> D[Tasks over partitions]
  D --> E[Action result or files]
```

Create a local SparkSession, inspect a small DataFrame, and notice that `show`, `count`, and `explain` trigger or describe execution.


In [2]:
from pathlib import Path
import sys
import os
# Make the local src-layout package importable in VS Code/Jupyter without installation.
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(project_root / "src"))
# Spark 3.5 is compatible with the installed Java 17 runtime; Java 25 is not.
java17_home = Path("/opt/miniconda3/pkgs/openjdk-17.0.18-h258754b_0/lib/jvm")
if java17_home.exists():
    os.environ["JAVA_HOME"] = str(java17_home)
    os.environ["PATH"] = f"{java17_home / 'bin'}:{os.environ['PATH']}"
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark_learning.spark_session import create_spark_session

spark = create_spark_session("Learning Lab: Getting Started")
raw = Path.cwd().parent / "data" / "raw"
if not raw.exists():
    raw = Path("data/raw")
orders = spark.read.option("header", True).csv(str(raw / "orders.csv"))
customers = spark.read.option("header", True).csv(str(raw / "customers.csv"))
orders.show(5, truncate=False)
orders.printSchema()


+--------+-----------+-------------------------+------+---------+-------------------------+
|order_id|customer_id|order_ts                 |amount|status   |ingested_at              |
+--------+-----------+-------------------------+------+---------+-------------------------+
|O000001 |C0063      |2025-01-27T10:00:00+00:00|358.26|paid     |2025-01-27T11:00:00+00:00|
|O000002 |C0070      |2025-01-11T09:00:00+00:00|229.05|paid     |2025-01-11T10:00:00+00:00|
|O000003 |C0065      |2025-01-27T20:00:00+00:00|18.06 |paid     |2025-01-27T21:00:00+00:00|
|O000004 |C0087      |2025-02-10T03:00:00+00:00|21.98 |completed|2025-02-10T04:00:00+00:00|
|O000005 |C0006      |2025-01-14T23:00:00+00:00|334.23|shipped  |2025-01-15T00:00:00+00:00|
+--------+-----------+-------------------------+------+---------+-------------------------+
only showing top 5 rows

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- amount: string

## Guided example and output explanation

The result has one row per `status`: `count` counts non-null order IDs and `sum` adds valid numeric values. The formatted plan reveals scans, exchanges, and aggregate operators; execution is only demanded by `show()`.

**Common mistake:** treating a Python variable as local data after a transformation. It is a distributed plan until an action produces a result. **Performance:** select/filter early and use native functions so Spark can optimize work.


In [3]:
# Native expressions form one optimized plan; no large data is collected to Python.
typed_orders = (orders
    .withColumn("amount_num", F.col("amount").cast("double"))
    .withColumn("order_time", F.to_timestamp("order_ts"))
    .filter(F.col("amount_num") > 0))

summary = typed_orders.groupBy("status").agg(
    F.count("order_id").alias("orders"),
    F.round(F.sum("amount_num"), 2).alias("revenue"),
).orderBy(F.desc("revenue"))
summary.show()
summary.explain("formatted")


+---------+------+--------+
|   status|orders| revenue|
+---------+------+--------+
|completed|   180| 37512.1|
|     paid|   153|31492.45|
|  shipped|   163|28777.72|
+---------+------+--------+

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [3]: [order_id#103, amount#106, status#107]
Batched: false
Location: InMemoryFileIndex [file:/Users/kirolousfouty/Downloads/Projects/pyspark/data/raw/orders.csv]
PushedFilters: [IsNotNull(amount)]
ReadSchema: struct<order_id:string,amount:string,status:string>

(2) Filter
Input [3]: [order_id#103, amount#106, status#107]
Condition : (isnotnull(amount#106) AND (cast(amount#106 as double) > 0.0))

(3) Project
Output [3]: [order_id#103, status#107, cast(amount#106 as double) AS amount_num#172]
Input [3]: [order_id#103, 

## Interactive experiment

Compare `df.filter('amount > 20').count()` with `df.filter('amount > 20').explain()`. Which line asks Spark for a result?

Change one parameter, run the cell, and inspect both output and plan. Record your prediction before running it.


In [4]:
# Experiment: partition count is observable, while an action materializes the plan.
print("Input partitions:", orders.rdd.getNumPartitions())
rebalanced = typed_orders.repartition(4, "customer_id")
print("Rebalanced partitions:", rebalanced.rdd.getNumPartitions())
print("Valid rows (action):", rebalanced.count())


Input partitions: 1
Rebalanced partitions: 4
Valid rows (action): 496


## Real-world lens

An e-commerce team commonly applies this pattern to raw order data before deriving a customer metric table. In production, inputs have explicit schemas, rejected records are observable, output is Parquet, and the transformation is covered by tests—see `src/pyspark_learning`.

## Practice exercises

1. Filter for completed orders, select `customer_id` and a numeric amount, and predict the schema before calling `printSchema()`.
2. Calculate revenue per customer, sort descending, and explain whether grouping can shuffle.
3. Join the customer lookup, preserving all valid orders; verify the row count and investigate any increase.

### Debugging challenge

This fails because `amount` is a string: `orders.filter(F.col('amount') > 10)`. Repair it without a Python loop and decide what to do with malformed values.

### Concept check

Which of `filter`, `withColumn`, `groupBy`, `show`, and `count` are actions? Which may create a shuffle, and why?


## Recap and readiness checklist

- I can state the business and execution purpose of this module's operation.
- I can predict whether my code remains lazy or requests a result.
- I can inspect schema, sample safely, and use `explain('formatted')`.
- I can name at least one correctness risk and one performance risk.

Solutions are in the matching file under `solutions/`. Stop the session when working outside Jupyter with `spark.stop()`.
